# 3 — Evolutionary search over CLIP token sequences

The goal is to find a fixed-length sequence of token IDs whose **CLIP text embedding** is close to a target image embedding. The decoded sequence may be awkward or non-human-readable; optimization happens in embedding space, not natural-language space.

A second stage feeds the resulting integer IDs into both SDXL text encoders and generates images with SDXL Turbo.

In [ ]:
# Requires an NVIDIA CUDA GPU.
# %pip install -e "..[all]"

## Genetic algorithm representation

- **Genome:** a fixed-length vector of CLIP vocabulary IDs.
- **Fitness:** cosine similarity between the target image embedding and each genome's CLIP text embedding.
- **Selection:** tournament selection.
- **Elitism:** copy the best fraction unchanged.
- **Variation:** one-point crossover, token swapping, and independent random token replacement.

The search is highly parallel: a population is scored in GPU batches.

In [ ]:
from clip_token_lab.evolution import EvolutionConfig

config = EvolutionConfig(
    n_tokens=16,
    population=1024,
    generations=100,
    score_batch_size=4096,
)
config

## Run image → optimized token IDs

Population size and score batch size are the main VRAM controls. Start much smaller when testing a new GPU. A deterministic run uses `random_seed=False` and a fixed `seed`.

In [ ]:
# from clip_token_lab.evolution import CLIPTokenOptimizer
# from clip_token_lab.io import load_rgb_image
#
# target = load_rgb_image("target.jpg")
# optimizer = CLIPTokenOptimizer()
# result = optimizer.optimize(
#     target,
#     config,
#     progress=lambda current, total, best: print(current, total, best),
# )
# result

## Generate from optimized IDs

This is an experimental vocabulary transfer. CLIP and SDXL tokenizers are related, but the meaning of a numeric ID is not guaranteed to be identical across every tokenizer. Inspect the SDXL-side token preview before interpreting the sequence.

In [ ]:
# from clip_token_lab.sdxl_tokens import SDXLTokenGenerator
#
# images = SDXLTokenGenerator().generate(
#     result.token_ids,
#     seed=0,
#     steps=2,
#     guidance_scale=0.0,
#     count=4,
# )

## Launch the two-stage UI

The UI serializes heavy GPU jobs with a queue. Model creation is lazy, but running both CLIP-L/14 and SDXL can still require substantial VRAM.

In [ ]:
from clip_token_lab.apps.evolution import build_demo

demo = build_demo()
demo.launch(inline=True)

## Minimal scripts

- `scripts/evolution/image_to_tokens.py` writes a JSON result containing decoded text, score, and token IDs.
- `scripts/evolution/tokens_to_image.py` reads that JSON and generates an image grid.

Keeping the two stages separate makes it possible to archive, compare, and regenerate evolved sequences without rerunning the genetic search.